<a href="https://www.kaggle.com/code/dozeradi007/dual-image-sr-dataset-creation?scriptVersionId=337782093" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [8]:
import os
import rasterio
from rasterio.enums import Resampling
from affine import Affine
import random

## Making directory structure
- Creates a dataset with the following structure:
```
    output_dir/ 
        train/
            HR_0.5m/
            LR_1m/
            LR_2m/
        val/
            HR_0.5m/
            LR_1m/
            LR_2m/
        test/
            HR_0.5m/
            LR_1m/
            LR_2m/
```
- Images are renamed continuously:
    - img_001.tif
    - img_002.tif

In [5]:
def create_dataset(input_dir, output_dir, train_ratio=0.7, 
                   val_ratio=0.15, test_ratio=0.15, seed=42, file_count=250):

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    hr_dir = os.path.join(input_dir, "HR_0.5m/HR_0.5m")
    # lr1_dir = os.path.join(input_dir, "LR_1m")
    lr2_dir = os.path.join(input_dir, "LR_2m/LR_2m")

    # -------------------------
    # Check input folders
    # -------------------------
    for folder in [hr_dir, lr2_dir]:
        if not os.path.isdir(folder):
            raise FileNotFoundError(f"{folder} not found.")

    hr_files = sorted(os.listdir(hr_dir))[:file_count]
    # lr1_files = sorted(os.listdir(lr1_dir))[:file_count]
    lr2_files = sorted(os.listdir(lr2_dir))[:file_count]

    #safety check
    if hr_files != lr2_files:
        raise ValueError("The filenames in HR_0.5m, LR_1m and LR_2m do not match.")
    print("Checked input folders")

    filenames = hr_files.copy()

    # -------------------------
    # Shuffle once
    # -------------------------
    random.seed(seed)
    random.shuffle(filenames)

    total = len(filenames)

    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    splits = {
        "train": filenames[:train_end],
        "val": filenames[train_end:val_end],
        "test": filenames[val_end:]
    }

    # -------------------------
    # Create folders
    # -------------------------
    for split in ["train", "val", "test"]:
        for folder in ["HR_0.5m", "LR_2m"]:
            os.makedirs(os.path.join(output_dir, split, folder), exist_ok=True)
    print("Folders Created")

    # -------------------------
    # Copy files
    # -------------------------
    counter = 1

    for split in ["train", "val", "test"]:

        for filename in splits[split]:

            new_name = f"img_{counter:03d}.tif"
            
            copy_files(os.path.join(hr_dir, filename), 
                       os.path.join(output_dir, split, "HR_0.5m", new_name))
            
            copy_files(os.path.join(lr2_dir, filename), 
                       os.path.join(output_dir, split, "LR_2m", new_name))

            counter += 1
        print(f"Copied {split} images")

    print("Dataset created successfully!\n")
    print(f"Train : {len(splits['train'])}")
    print(f"Val   : {len(splits['val'])}")
    print(f"Test  : {len(splits['test'])}")
    print(f"Total : {total}")

In [6]:
def copy_files(input_path,output_path,file_count=250):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with rasterio.open(input_path) as src:
        img=src.read()
        profile=src.profile
        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(img)

In [9]:
create_dataset(
    input_dir="/kaggle/input/4x-satellite-image-super-resolution/",
    output_dir="/kaggle/working/dataset",
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42
)

Dataset created successfully!

Train : 175
Val   : 37
Test  : 38
Total : 250


## Generating LR images at 1m scale (downgrading HR images)

In [10]:
def downgrade_img(data_dir, scale_factor=2):
    # os.makedirs(lr_dir, exist_ok=True)
    for split in ['train', 'test', 'val']:
        hr_dir=os.path.join(data_dir, split, "HR_0.5m")
        split_lr_dir = os.path.join(data_dir, split, "LR_1m")
        os.makedirs(split_lr_dir, exist_ok=True)
        files=sorted(os.listdir(hr_dir))

        for f in files:
            hr_path=os.path.join(hr_dir,f)
            lr_path=os.path.join(split_lr_dir,f)
            # --- Read the HR image ---
            with rasterio.open(hr_path) as hr_ds:
                # Calculate new shape
                new_height = hr_ds.height // scale_factor
                new_width = hr_ds.width // scale_factor
                
                # Read and resample the image
                lr_data = hr_ds.read(
                    out_shape=(
                        hr_ds.count,
                        new_height,
                        new_width
                    ),
                    resampling=Resampling.bilinear  # or Resampling.cubic for smoother results
                )
            
                # Compute new transform (georeferencing adjustment)
                new_transform = hr_ds.transform * Affine.scale(
                    hr_ds.width / new_width,
                    hr_ds.height / new_height
                )

                # --- Save the LR image ---
                profile = hr_ds.profile
                profile.update({
                    'height': new_height,
                    'width': new_width,
                    'transform': new_transform
                })
                # lr_path=os.path.join(lr_dir,"austin1_2x.tif")
            
                with rasterio.open(lr_path, 'w', **profile) as lr_ds:
                    lr_ds.write(lr_data)
        print(f"Downgraded {split} HR images")

In [11]:
downgrade_img("/kaggle/working/dataset", scale_factor=2)

Downgraded train HR images
Downgraded test HR images
Downgraded val HR images
